In [1]:
import sqlite3

# create tables
conn = sqlite3.connect("student_db.sqlite")
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS Enrollment;
DROP TABLE IF EXISTS Course;
DROP TABLE IF EXISTS Student;

CREATE TABLE Student (
    StudentID INT PRIMARY KEY,
    StudentName VARCHAR(100)
);

CREATE TABLE Course (
    CourseID INT PRIMARY KEY,
    CourseName VARCHAR(100),
    Credits INT
);

CREATE TABLE Enrollment (
    StudentID INT,
    CourseID INT,
    PRIMARY KEY (StudentID, CourseID),
    FOREIGN KEY (StudentID) REFERENCES Student(StudentID),
    FOREIGN KEY (CourseID) REFERENCES Course(CourseID)
);
""")

conn.commit()
conn.close()

In [2]:
import sqlite3

conn = sqlite3.connect("student_db.sqlite")
cur = conn.cursor()

# Insert at least 5 students
students = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Carla"),
    (4, "David"),
    (5, "Eva")
]
cur.executemany(
    "INSERT INTO Student(StudentID, StudentName) VALUES (?, ?);",
    students
)

# Insert at least 5 courses
courses = [
    (101, "Math 101", 3),
    (102, "CS 101", 4),
    (103, "History 101", 3),
    (104, "Biology 101", 4),
    (105, "Art 101", 2)
]
cur.executemany(
    "INSERT INTO Course(CourseId, CourseName, Credits) VALUES (?, ?, ?);",
    courses
)

# Insert at least 10 enrollments
enrollments = [
    (1, 101), (1, 102),
    (2, 101), (2, 103),
    (3, 102), (3, 104),
    (4, 101), (4, 105),
    (5, 103), (5, 104)
]
cur.executemany(
    "INSERT INTO Enrollment(StudentID, CourseID) VALUES (?, ?);",
    enrollments
)

conn.commit()
conn.close()

In [3]:
import sqlite3

def get_connection():
    return sqlite3.connect("student_db.sqlite")

def get_or_create_student():
    sid = int(input("Enter your StudentID (or -1 for new): "))
    conn = get_connection()
    cur = conn.cursor()

    if sid == -1:
        name = input("Enter new student name: ")
        cur.execute("SELECT COALESCE(MAX(StudentID), 0) + 1 FROM Student;")
        sid = cur.fetchone()[0]
        cur.execute(
            "INSERT INTO Student(StudentID, StudentName) VALUES (?, ?);",
            (sid, name)
        )
        conn.commit()
        print(f"Created student {name} with ID {sid}")
    else:
        cur.execute("SELECT StudentName FROM Student WHERE StudentID = ?;", (sid,))
        row = cur.fetchone()
        if row is None:
            print("No such student; creating a new one.")
            name = input("Enter student name: ")
            cur.execute(
                "INSERT INTO Student(StudentID, StudentName) VALUES (?, ?);",
                (sid, name)
            )
            conn.commit()
        else:
            print(f"Welcome, {row[0]}!")

        conn.close()
        return sid

def list_courses():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT CourseID, CourseName, Credits FROM Course;")
    for cid, name, credits in cur.fetchall():
        print(f"{cid}: {name} ({credits}) credits)")
    conn.close()

def enroll(student_id):
    conn = get_connection()
    cur = conn.cursor()
    raw = input("Enter CourseID to enroll: ")
    if not raw.isdigit():
        print("CourseID must be a number.")
        conn.close()
        return
        
    course_id = int(raw)
    cur.execute(
        "SELECT 1 FROM Enrollment WHERE StudentID = ? AND CourseID = ?;",
        (student_id, course_id)
    )
    if cur.fetchone():
        print("Already enrolled in this course.")
    else:
        cur.execute(
            "INSERT INTO Enrollment(StudentID, CourseID) VALUES (?, ?);",
            (student_id, course_id)
        )
        conn.commit()
        print("Enrolled successfully.")
    conn.close()

def withdraw(student_id):
    conn = get_connection()
    cur = conn.cursor()
    raw = input("Enter CourseID to withdraw from: ")
    if not raw.isdigit():
        print("CourseID must be a number.")
        conn.close()
        return
        
    course_id = int(raw)
    cur.execute(
        "DELETE FROM Enrollment WHERE StudentID = ? AND CourseID = ?;",
        (student_id, course_id)
    )
    conn.commit()
    print("Withdrawal complete (if enrollment existed).")
    conn.close()

def search_courses():
    conn = get_connection()
    cur = conn.cursor()
    substring = input("Enter part of course name to search: ")
    cur.execute(
        "SELECT CourseID, CourseName, Credits FROM Course "
        "WHERE CourseName LIKE ?;",
        (f"%{substring}%",)
    )
    rows = cur.fetchall()
    if not rows:
        print("No matching courses.")
    else:
        for cid, name, credits in rows:
            print(f"{cid}: {name} ({credits} credits)")
        conn.close()

def my_classes(student_id):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        SELECT c.CourseID, c.CourseName, c.Credits
        FROM Course c
        JOIN Enrollment e ON c.CourseID = e.CourseID
        WHERE e.StudentID = ?;
    """, (student_id,))
    rows = cur.fetchall()
    if not rows:
        print("You are not enrolled in any classes.")
    else:
        for cid, name, credits in rows:
            print(f"{cid}: {name} ({credits} credits)")
    conn.close()

def main():
    student_id = get_or_create_student()
    while True:
        print("\nStudent Menu")
        print("L - List all courses")
        print("E - Enroll in a course")
        print("W - Withdraw from a course")
        print("S - Search courses by name")
        print("M - My Classes")
        print("X - Exit")
        choice = input("Choose an option: ").strip().upper()

        if choice == "L":
            list_courses()
        elif choice == "E":
            enroll(student_id)
        elif choice == "W":
            withdraw(student_id)
        elif choice == "S":
            search_courses()
        elif choice == "M":
            my_classes(student_id)
        elif choice == "X":
            print("Goodbye!")
            break
        else:
            print("Invalid option, try again.")

main()

Enter your StudentID (or -1 for new):  1


Welcome, Alice!

Student Menu
L - List all courses
E - Enroll in a course
W - Withdraw from a course
S - Search courses by name
M - My Classes
X - Exit


Choose an option:  L


101: Math 101 (3) credits)
102: CS 101 (4) credits)
103: History 101 (3) credits)
104: Biology 101 (4) credits)
105: Art 101 (2) credits)

Student Menu
L - List all courses
E - Enroll in a course
W - Withdraw from a course
S - Search courses by name
M - My Classes
X - Exit


Choose an option:  E
Enter CourseID to enroll:  M


CourseID must be a number.

Student Menu
L - List all courses
E - Enroll in a course
W - Withdraw from a course
S - Search courses by name
M - My Classes
X - Exit


Choose an option:  W
Enter CourseID to withdraw from:  W


CourseID must be a number.

Student Menu
L - List all courses
E - Enroll in a course
W - Withdraw from a course
S - Search courses by name
M - My Classes
X - Exit


Choose an option:  X


Goodbye!
